## 1. Verify Pre-installed Packages

In [ ]:
import importlib.metadata

packages = [
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-aws",
    "langchain-mcp-adapters",
    "mcp",
    "httpx",
    "boto3",
]

print("Pre-installed packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
# Install compatible versions
%pip install -U "langgraph>=0.2,<1.0" "langchain-core>=0.3,<1.0" "langchain-aws>=0.2" "langchain-mcp-adapters>=0.2.1" nest-asyncio -q

## 2. Imports

In [ ]:
import asyncio
from datetime import timedelta

import nest_asyncio
nest_asyncio.apply()

from langchain_aws import ChatBedrockConverse
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

print("All imports successful!")

## 3. Configuration

Set `MODEL` and your MCP Gateway credentials.

In [ ]:
#################################################
# CONFIGURATION
#################################################

MODEL = "sonnet4"  # Options: haiku, sonnet, sonnet4, sonnet45

# MCP Gateway credentials (from .mcp-credentials.json)
GATEWAY_URL = "YOUR_GATEWAY_URL_HERE"
ACCESS_TOKEN = "YOUR_ACCESS_TOKEN_HERE"

#################################################

# Model configurations (auto-derived from MODEL)
MODELS = {
    "haiku": {
        "id": "anthropic.claude-3-5-haiku-20241022-v1:0",
        "profile": "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    },
    "sonnet": {
        "id": "anthropic.claude-3-5-sonnet-20241022-v2:0",
        "profile": "us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    },
    "sonnet4": {
        "id": "anthropic.claude-sonnet-4-20250514-v1:0",
        "profile": "us.anthropic.claude-sonnet-4-20250514-v1:0",
    },
    "sonnet45": {
        "id": "anthropic.claude-sonnet-4-5-20250929-v1:0",
        "profile": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    },
}

# Set model variables (always)
if MODEL not in MODELS:
    raise ValueError(f"Unknown MODEL '{MODEL}'. Valid: {list(MODELS.keys())}")

BASE_MODEL_ID = MODELS[MODEL]["id"]
MODEL_ID = MODELS[MODEL]["profile"]
REGION = "us-west-2"

print(f"Model:   {MODEL}")
print(f"Using:   {MODEL_ID}")

# Validate gateway credentials
if "YOUR_" in GATEWAY_URL or "YOUR_" in ACCESS_TOKEN:
    print("\nWARNING: Set GATEWAY_URL and ACCESS_TOKEN before running MCP cells")
else:
    print(f"Gateway: {GATEWAY_URL[:50]}...")
    print("Configuration OK!")

## 4. System Prompt

In [ ]:
SYSTEM_PROMPT = """You are a helpful Neo4j database assistant with access to tools that let you query a Neo4j graph database.

Your capabilities include:
- Retrieve the database schema to understand node labels, relationship types, and properties
- Execute read-only Cypher queries to answer questions about the data
- Do not execute any write Cypher queries

When answering questions about the database:
1. First retrieve the schema to understand the database structure
2. Formulate appropriate Cypher queries based on the actual schema
3. If a query returns no results, explain what you looked for and suggest alternatives
4. Format results in a clear, human-readable way
5. Cite the actual data returned in your response

Important Cypher notes:
- Use MATCH patterns that align with the actual schema
- For counting, use MATCH (n:Label) RETURN count(n)
- For listing items, add LIMIT to avoid overwhelming results
- Handle potential NULL values gracefully

Be concise but thorough in your responses."""

## 5. Initialize LLM

In [ ]:
llm = ChatBedrockConverse(
    model=MODEL_ID,
    provider="anthropic",
    region_name=REGION,
    temperature=0,
    base_model_id=BASE_MODEL_ID,
)

print(f"LLM initialized with {MODEL}!")

## 6. Query Helper

In [ ]:
async def query_async(question: str) -> str:
    """Ask the agent a question about the Neo4j database."""
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    
    async with streamablehttp_client(
        GATEWAY_URL,
        headers,
        timeout=timedelta(seconds=120),
        terminate_on_close=False
    ) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            
            agent = create_react_agent(
                model=llm,
                tools=tools,
                prompt=SYSTEM_PROMPT,
            )
            
            result = await agent.ainvoke({"messages": [("human", question)]})
            messages = result.get("messages", [])
            return getattr(messages[-1], "content", str(messages[-1])) if messages else "No response"


def query(question: str) -> str:
    """Ask the agent a question about the Neo4j database."""
    print("=" * 70)
    print(f"Q: {question}")
    print("=" * 70)
    answer = asyncio.get_event_loop().run_until_complete(query_async(question))
    print(f"\nA: {answer}")
    return answer

## 7. Demo Queries

In [ ]:
_ = query("What is the database schema? Give me a brief summary.")

In [ ]:
_ = query("How many nodes are in the database by label?")

In [ ]:
_ = query("What types of relationships exist in the database?")

## 8. Your Queries

In [ ]:
_ = query("List 5 sample records from the most populated node type.")

In [ ]:
# Your custom query
# _ = query("Your question here")